In [1]:
import pandas as pd
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the datasets
train_df = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv')
test_df = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/test.csv')

# Copy the datasets to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Identify categorical and numerical columns
categorical_features = train_df.select_dtypes(include=['object']).columns.tolist()
numerical_features = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Handle missing values
fill_missing_value = FillMissingValue(features=numerical_features, strategy='mean')
train_df_copy = fill_missing_value.fit_transform(train_df_copy)
test_df_copy = fill_missing_value.transform(test_df_copy)

# Encode categorical variables
label_encode = LabelEncode(features=categorical_features)
train_df_copy = label_encode.fit_transform(train_df_copy)
test_df_copy = label_encode.transform(test_df_copy)

# Scale numerical features
standard_scale = StandardScale(features=numerical_features)
train_df_copy = standard_scale.fit_transform(train_df_copy)
test_df_copy = standard_scale.transform(test_df_copy)

# Display the preprocessed datasets
train_df_copy.head(), test_df_copy.head()


2025-08-31 11:15:29.058 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


(         id  CustomerId  Surname  ...  IsActiveMember  EstimatedSalary    Exited
 0  1.403375    1.236497     2703  ...        1.005415        -0.181454 -0.518407
 1  1.725899   -0.169062     2302  ...       -0.994614        -0.193591 -0.518407
 2  1.533110   -0.758615     1510  ...       -0.994614         0.849538 -0.518407
 3  0.877727    0.516472      888  ...        1.005415        -0.104529 -0.518407
 4  0.536127   -1.094916     1505  ...       -0.994614         0.426198 -0.518407
 
 [5 rows x 14 columns],
          id  CustomerId  Surname  ...  IsActiveMember  EstimatedSalary    Exited
 0 -1.035324    0.848248      519  ...       -0.994614        -0.658497 -0.518407
 1 -0.966401    0.712439     1949  ...       -0.994614        -1.481697 -0.518407
 2 -0.481839    0.540344     2026  ...       -0.994614        -0.867841  1.928985
 3  0.206392    1.562599      546  ...       -0.994614        -1.001205 -0.518407
 4  1.348244    0.608613     1130  ...       -0.994614        -0.425216 

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info_train = get_column_info(train_df_copy)
print("column_info_train")
print(column_info_train)

column_info_test = get_column_info(test_df_copy)
print("column_info_test")
print(column_info_test)


column_info_train
{'Category': [], 'Numeric': ['id', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}
column_info_test
{'Category': [], 'Numeric': ['id', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

# Define the target and features
target = 'Exited'
features = [col for col in train_df_copy.columns if col not in [target, 'id', 'CustomerId', 'Surname']]

# Initialize the Random Forest model with hyperparameters
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, random_state=42)

# Train the model
rf_model.fit(train_df_copy[features], train_df_copy[target])

# Predict on the test set
test_predictions = rf_model.predict(test_df_copy[features])
test_probabilities = rf_model.predict_proba(test_df_copy[features])[:, 1]

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(test_df_copy[target], test_probabilities)

# Calculate the confusion matrix
cm = confusion_matrix(test_df_copy[target], test_predictions)

# Display the AUC-ROC score and confusion matrix
print(f"AUC-ROC Score: {auc_roc}")
print(f"Confusion Matrix:\n{cm}")

# Save the predictions
test_df_copy['PredictedExited'] = test_predictions
test_df_copy[['id', 'PredictedExited']].to_csv('predictions.csv', index=False)


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.